# 03 — Synthetic constellation situational view

The upstream God's Eye View combines many public signals into one global situational picture, including a satellite layer. This original notebook builds a small **synthetic constellation** and computes all sub-satellite points at a single instant. It is designed to be fast enough for a browser-only Pyodide kernel.


In [ ]:
import numpy as np
import matplotlib.pyplot as plt
MU = 398600.4418
R_E = 6378.137
OMEGA_E = 7.2921159e-5

def subpoint(alt_km, inc_deg, raan_deg, phase_deg, t_s):
    a = R_E + alt_km; n = np.sqrt(MU/a**3)
    i, raan, u = np.deg2rad([inc_deg, raan_deg, phase_deg])
    u = u + n*t_s
    x = a*(np.cos(raan)*np.cos(u)-np.sin(raan)*np.sin(u)*np.cos(i))
    y = a*(np.sin(raan)*np.cos(u)+np.cos(raan)*np.sin(u)*np.cos(i))
    z = a*np.sin(u)*np.sin(i)
    th = OMEGA_E*t_s
    xe = np.cos(th)*x + np.sin(th)*y
    ye = -np.sin(th)*x + np.cos(th)*y
    return np.rad2deg(np.arctan2(z, np.hypot(xe,ye))), np.rad2deg(np.arctan2(ye,xe))

# 6 orbital planes × 8 satellites, synthetic Walker-like layout.
records=[]
for plane in range(6):
    for slot in range(8):
        records.append((550.0, 53.0, plane*60.0, slot*45.0 + plane*7.5, plane))

t_s = 37*60
pts = np.array([subpoint(*r[:4], t_s) for r in records])
planes = np.array([r[4] for r in records])
print(f"Synthetic satellites: {len(records)}")


In [ ]:
fig, ax = plt.subplots(figsize=(11,5))
for p in np.unique(planes):
    m = planes == p
    ax.scatter(pts[m,1], pts[m,0], s=30, label=f"plane {p+1}")
ax.set(xlim=(-180,180), ylim=(-90,90), xlabel="Longitude (deg)", ylabel="Latitude (deg)", title="Synthetic constellation sub-satellite points at t = 37 min")
ax.set_xticks(np.arange(-180,181,60)); ax.set_yticks(np.arange(-90,91,30)); ax.grid(True, alpha=.3)
ax.legend(ncol=3, fontsize=8); plt.show()


In [ ]:
# A simple 30°×30° occupancy matrix gives a compact 'where is coverage concentrated?' summary.
lat_edges = np.arange(-90, 91, 30)
lon_edges = np.arange(-180, 181, 30)
H, _, _ = np.histogram2d(pts[:,0], pts[:,1], bins=[lat_edges, lon_edges])
fig, ax = plt.subplots(figsize=(10,4.5))
im = ax.imshow(H, origin="lower", aspect="auto", extent=[-180,180,-90,90])
ax.set(xlabel="Longitude (deg)", ylabel="Latitude (deg)", title="30° × 30° satellite-count grid")
fig.colorbar(im, ax=ax, label="satellites"); plt.show()


## From demo to real orbital data

For a real application, replace the synthetic orbital elements with an authorized source, preserve the provider's attribution/terms, and use a validated propagator such as SGP4. God's Eye View documents **CelesTrak** as its satellite TLE source and credits “CelesTrak (celestrak.org), Dr. T.S. Kelso.”
